In [1]:
import re
import pandas as pd
from google.cloud import bigquery
import openpyxl
import os
import glob

# Helper function to clean illegal characters from a string
def clean_string(value):
    if isinstance(value, str):
        # Remove non-printable characters (including ASCII control characters)
        return re.sub(r'[\x00-\x1F\x7F]', '', value)
    return value

# Function to clean specific columns in the dataframe
def clean_dataframe(df, columns):
    for column in columns:
        if column in df.columns:
            df[column] = df[column].apply(clean_string)
    return df

# Function to find a column by its header name (case-insensitive)
def find_column_by_header(sheet, header_name):
    header_name_normalized = header_name.strip().lower()  # Normalize input header name
    for col in range(1, sheet.max_column + 1):
        cell_value = sheet.cell(row=1, column=col).value
        if cell_value and cell_value.strip().lower() == header_name_normalized:
            return col
    raise ValueError(f"Header '{header_name}' not found in the sheet.")

# Function to automatically find the SDP Input Excel file
def find_sdp_input_file():
    sdp_files = glob.glob("*SDP Input*.xlsx")
    if not sdp_files:
        raise FileNotFoundError("No SDP Input file found.")
    return sdp_files[0]

# Function to automatically find the unmatched CSV file
def find_unmatched_csv_file():
    unmatched_files = glob.glob("unmatched_export*.csv")
    if not unmatched_files:
        raise FileNotFoundError("No unmatched_export CSV file found.")
    return unmatched_files[0]

# Prompt for the BigQuery 16-character inputs
def get_bigquery_keys():
    key1 = input("Enter the first 16-character key for BigQuery: ")
    key2 = input("Enter the second 16-character key for BigQuery: ")
    return key1, key2

# Function to retrieve backend data from BigQuery
def get_backend_data(key1, key2):
    client = bigquery.Client(project='tealbook-prd')  # Specify your GCP project ID
    query = f"""
    SELECT
      DISTINCT(r.internalSupplierId) AS internalSupplierId,
      ARRAY_TO_STRING(og.domains,"|") AS domains,
      og.name,
      og.supplier.locations[SAFE_OFFSET(0)].address as Primary_Address
    FROM
        `tealbook-prd.tealbook.RelationshipSpendLineItems` r
    LEFT JOIN
      `tealbook-prd.tealbook.OrgUnit` og
    ON
      SUBSTR(r.supplier, 9, 16) = SUBSTR(og.__key__.path, 8,16)
      AND RIGHT(r.supplier, 16) = SUBSTR(og.__key__.path, 37,16)
    WHERE
        ARRAY_TO_STRING(og.domains,"|") != "missing.link"
        AND SUBSTR(r.__key__.path, 8,16)= "{key1}"
        AND SUBSTR(r.__key__.path, 37,16)= "{key2}";
    """
    query_job = client.query(query)
    return query_job.to_dataframe()

# Function to process and create the output file
# Modify the create_output_file function
def create_output_file(sdp_input_path, unmatched_csv_path, backend_data):
    # Read SDP Input and unmatched CSV files
    sdp_input = pd.read_excel(sdp_input_path, engine='openpyxl')
    unmatched_data = pd.read_csv(unmatched_csv_path)
    
    # Clean illegal characters in backend data
    backend_data = clean_dataframe(backend_data, ['Primary_Address', 'name', 'domains'])

    # Create Excel writer
    output_filename = f"{os.path.splitext(sdp_input_path)[0]} - primary addresses.xlsx"
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # Write tabs: Vendor Master, unmatched, and backend
        sdp_input.to_excel(writer, sheet_name='Vendor Master', index=False)
        unmatched_data[['internal_supplier_id']].to_excel(writer, sheet_name='unmatched', index=False)
        backend_data.to_excel(writer, sheet_name='backend', index=False)
    
    # Open the saved workbook to modify Vendor Master tab
    workbook = openpyxl.load_workbook(output_filename)
    vendor_master = workbook['Vendor Master']

    # Add the "Unmatched" column and populate it based on unmatched_data
    unmatched_col = vendor_master.max_column + 1
    vendor_master.cell(row=1, column=unmatched_col).value = "Unmatched"
    unmatched_ids = unmatched_data['internal_supplier_id'].astype(str).str.strip().tolist()
    
    backend_df = pd.DataFrame(backend_data)
    complete_address_col = find_column_by_header(vendor_master, "Complete Address")

    # Step 1: Populate Unmatched column based on unmatched_ids
    for row in range(2, vendor_master.max_row + 1):
        internal_supplier_id = str(vendor_master.cell(row=row, column=1).value).strip()
        if internal_supplier_id in unmatched_ids:
            vendor_master.cell(row=row, column=unmatched_col).value = "no match"

    # Step 2: Replace "no match" with Primary_Address from backend data
    for row in range(2, vendor_master.max_row + 1):
        unmatched_value = vendor_master.cell(row=row, column=unmatched_col).value
        if unmatched_value == "no match":
            internal_supplier_id = str(vendor_master.cell(row=row, column=1).value).strip()
            match = backend_df[backend_df['internalSupplierId'] == internal_supplier_id]
            if not match.empty:
                primary_address = match['Primary_Address'].values[0]
                vendor_master.cell(row=row, column=unmatched_col).value = primary_address

    # Step 3: Replace "Complete Address" if Unmatched column is populated
    for row in range(2, vendor_master.max_row + 1):
        unmatched_value = vendor_master.cell(row=row, column=unmatched_col).value
        if unmatched_value and unmatched_value != "no match":
            vendor_master.cell(row=row, column=complete_address_col).value = unmatched_value

    # Step 4: Populate "web_domain" column with corresponding "domains" value from backend
    try:
        web_domain_col = find_column_by_header(vendor_master, "web_domain")
    except ValueError:
        web_domain_col = vendor_master.max_column + 1
        vendor_master.cell(row=1, column=web_domain_col).value = "web_domain"
    
    # Update or populate web_domain column based on backend domains
    for row in range(2, vendor_master.max_row + 1):
        internal_supplier_id = str(vendor_master.cell(row=row, column=1).value).strip()
        match = backend_df[backend_df['internalSupplierId'] == internal_supplier_id]
        
        if not match.empty:
            web_domain_value = match['domains'].values[0]
            vendor_master.cell(row=row, column=web_domain_col).value = web_domain_value

    # Step 5: Remove any "web_domain" values containing "missing.link" (case-insensitive)
    for row in range(2, vendor_master.max_row + 1):
        web_domain_value = str(vendor_master.cell(row=row, column=web_domain_col).value).strip().lower()
        if "missing.link" in web_domain_value:
            vendor_master.cell(row=row, column=web_domain_col).value = None  # Clear cell value

    # Save the workbook with the updated information
    workbook.save(output_filename)

# Main function
def main():
    try:
        # Automatically find the SDP Input and unmatched CSV files
        sdp_input_path = find_sdp_input_file()
        unmatched_csv_path = find_unmatched_csv_file()
        
        print(f"SDP Input file found: {sdp_input_path}")
        print(f"Unmatched CSV file found: {unmatched_csv_path}")
        
        # Get the BigQuery keys and fetch the backend data
        key1, key2 = get_bigquery_keys()
        backend_data = get_backend_data(key1, key2)
        
        # Create the output file
        create_output_file(sdp_input_path, unmatched_csv_path, backend_data)
        print(f"Output file created successfully: {os.path.splitext(sdp_input_path)[0]} - primary addresses.xlsx")
    
    except FileNotFoundError as e:
        print(e)

# Uncomment to run the main function
if __name__ == "__main__":
    main()


SDP Input file found: Step 1_Hilton - SDP Input - 2024-11-05.xlsx
Unmatched CSV file found: unmatched_export_f9293a37-3fbd-445f-87c4-c869ef708e7c_2024-11-05 19_11_03.990812+00_00.csv


Enter the first 16-character key for BigQuery:  5068991043207168
Enter the second 16-character key for BigQuery:  5629499534213120


Output file created successfully: Step 1_Hilton - SDP Input - 2024-11-05 - primary addresses.xlsx
